In [ ]:
# Run this if you have GPU
# !pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
# !pip install ultralytics tensorflow keras scikit-learn numpy pandas matplotlib seaborn pillow

In [ ]:
# Run this if you have CPU
# !pip install torch torchvision torchaudio ultralytics tensorflow keras scikit-learn numpy pandas matplotlib seaborn pillow

# Skin Disease Screening Tool Using AI - Senior Design Project 

## Imports

### Default Imports

In [3]:
import os
import time
import copy
import shutil
import random
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from PIL import Image

### AI/ML/CNN Imports

In [8]:
# PyTorch
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader

# TensorFlow / Keras
import tensorflow as tf
import keras

# Ultralytics (YOLO)
from ultralytics import YOLO

Creating new Ultralytics Settings v0.0.6 file  
View Ultralytics Settings with 'yolo settings' or at 'C:\Users\Ali\AppData\Roaming\Ultralytics\settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


## Loading Dataset

In [ ]:
# Dataset Formatting and Splitting
source_dir = 'skin-diseases-image-dataset/IMG_CLASSES'
yolo_dir = 'yolo_skin_dataset'

# Remove the hidden Jupyter checkpoints folder if it exists to prevent FileNotFoundError
checkpoints_dir = os.path.join(source_dir, '.ipynb_checkpoints')
if os.path.exists(checkpoints_dir):
    shutil.rmtree(checkpoints_dir)

# Find classes
classes = [d for d in os.listdir(source_dir) if os.path.isdir(os.path.join(source_dir, d))]

for cls in classes:
    os.makedirs(os.path.join(yolo_dir, 'train', cls), exist_ok=True)
    os.makedirs(os.path.join(yolo_dir, 'val', cls), exist_ok=True)
    
    cls_path = os.path.join(source_dir, cls)
    images = [f for f in os.listdir(cls_path) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
    random.shuffle(images)
    
    # 80/20 Split
    split_idx = int(0.8 * len(images))
    train_imgs, val_imgs = images[:split_idx], images[split_idx:]
    
    for img in train_imgs:
        shutil.copy(os.path.join(cls_path, img), os.path.join(yolo_dir, 'train', cls, img))
    for img in val_imgs:
        shutil.copy(os.path.join(cls_path, img), os.path.join(yolo_dir, 'val', cls, img))

print(f"Dataset securely formatted in '{yolo_dir}'")


## Model Training

### ResNet

In [ ]:
# PyTorch Data Loading and ResNet Setup
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# 1. Heavy Augmentations to combat overfitting
train_transforms = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5), 
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

val_transforms = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# 2. Load Datasets
train_dataset = datasets.ImageFolder(os.path.join(yolo_dir, 'train'), transform=train_transforms)
val_dataset = datasets.ImageFolder(os.path.join(yolo_dir, 'val'), transform=val_transforms)

dataloaders = {
    'train': DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2),
    'val': DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)
}
class_names = train_dataset.classes

# 3. Model Setup
resnet_model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
num_ftrs = resnet_model.fc.in_features
resnet_model.fc = nn.Linear(num_ftrs, len(class_names))
resnet_model = resnet_model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer_resnet = optim.Adam(resnet_model.parameters(), lr=0.001)

# PyTorch Training Execution
def train_model(model, criterion, optimizer, num_epochs=10):
    since = time.time()
    best_model_wts = copy.deepcopy(model.state_dict())
    best_acc = 0.0

    for epoch in range(num_epochs):
        print(f'Epoch {epoch+1}/{num_epochs}')
        print('-' * 10)

        for phase in ['train', 'val']:
            if phase == 'train':
                model.train()
            else:
                model.eval()

            running_loss = 0.0
            running_corrects = 0

            for inputs, labels in dataloaders[phase]:
                inputs, labels = inputs.to(device), labels.to(device)
                optimizer.zero_grad()

                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)

                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)

            epoch_loss = running_loss / len(dataloaders[phase].dataset)
            epoch_acc = running_corrects.double() / len(dataloaders[phase].dataset)

            print(f'{phase.capitalize()} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')

            if phase == 'val' and epoch_acc > best_acc:
                best_acc = epoch_acc
                best_model_wts = copy.deepcopy(model.state_dict())
        print()

    time_elapsed = time.time() - since
    print(f'Training complete in {time_elapsed // 60:.0f}m {time_elapsed % 60:.0f}s')
    print(f'Best Validation Accuracy: {best_acc:4f}')

    model.load_state_dict(best_model_wts)
    return model

# Execute ResNet training
trained_resnet = train_model(resnet_model, criterion, optimizer_resnet, num_epochs=15)


### YoloV8

In [ ]:
# Load the lightweight pre-trained YOLOv8 classification model
yolo_model = YOLO('yolov8n-cls.pt')

# Train the model pointing directly to your formatted root directory
# Augmentations are applied here to combat the overfitting you saw earlier
results = yolo_model.train(
    data='yolo_skin_dataset', 
    epochs=15, 
    imgsz=224,
    degrees=20.0,      # Apply random rotation up to 20 degrees
    fliplr=0.5,        # 50% chance to flip the image horizontallya
    flipud=0.5,        # 50% chance to flip the image vertically
    hsv_h=0.015,       # Slight hue adjustment
    hsv_s=0.7,         # Saturation adjustment
    hsv_v=0.4          # Value/Brightness adjustment
)

# Validate the model on the unseen validation split to test its true accuracy
metrics = yolo_model.val()
print(f"Top-1 Accuracy: {metrics.top1}")

# Load the best performing model
model_path = '/home/seniordesign02/runs/classify/train/weights/best.pt'
model = YOLO(model_path)


### EfficientNet

In [ ]:
yolo_dir = 'yolo_skin_dataset'
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

# 1. Apply the exact same heavy augmentations used in ResNet
train_transforms = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5), 
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

val_transforms = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# 2. Load Datasets
train_dataset = datasets.ImageFolder(os.path.join(yolo_dir, 'train'), transform=train_transforms)
val_dataset = datasets.ImageFolder(os.path.join(yolo_dir, 'val'), transform=val_transforms)

dataloaders = {
    'train': DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2),
    'val': DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)
}

# 3. Load PyTorch EfficientNetB0
efficientnet_model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)

# 4. Modify the final classification layer for your 5 diseases
# EfficientNet's final layer is inside a classifier block, taking 1280 input features
num_ftrs = efficientnet_model.classifier[1].in_features
efficientnet_model.classifier[1] = nn.Linear(num_ftrs, len(train_dataset.classes))
efficientnet_model = efficientnet_model.to(device)

# 5. Loss and Optimizer setup
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(efficientnet_model.parameters(), lr=0.001)

# 6. Execute Training using the train_model function you already loaded into memory
print("Starting PyTorch EfficientNet training...")
trained_efficientnet = train_model(efficientnet_model, criterion, optimizer, num_epochs=15)


## Models Evaluation

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

In [ ]:
def evaluate_model(model, dataloader, class_names, device, model_name="Model"):
    
    print(f"\n===== Evaluating {model_name} =====\n")
    
    model.eval()
    
    all_preds = []
    all_labels = []
    
    start_time = time.time()
    
    with torch.no_grad():
        for inputs, labels in dataloader:
            inputs = inputs.to(device)
            labels = labels.to(device)
            
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    inference_time = time.time() - start_time
    
    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)
    
    # ==============================
    # 1️⃣ Overall Accuracy
    # ==============================
    accuracy = np.mean(all_preds == all_labels)
    print(f"Overall Accuracy: {accuracy * 100:.2f}%")
    
    # ==============================
    # 2️⃣ Classification Report
    # ==============================
    print("\nClassification Report:")
    report = classification_report(all_labels, all_preds, 
                                   target_names=class_names, 
                                   output_dict=True)
    
    print(classification_report(all_labels, all_preds, target_names=class_names))
    
    # ==============================
    # 3️⃣ Confusion Matrix
    # ==============================
    cm = confusion_matrix(all_labels, all_preds)
    
    plt.figure()
    sns.heatmap(cm, annot=True, fmt="d",
                xticklabels=class_names,
                yticklabels=class_names)
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.title(f"{model_name} - Confusion Matrix")
    plt.show()
    
    # ==============================
    # 4️⃣ Specificity (per class)
    # ==============================
    print("\nSpecificity (per class):")
    specificity = {}
    
    for i, class_name in enumerate(class_names):
        TP = cm[i, i]
        FN = np.sum(cm[i, :]) - TP
        FP = np.sum(cm[:, i]) - TP
        TN = np.sum(cm) - (TP + FN + FP)
        
        spec = TN / (TN + FP) if (TN + FP) != 0 else 0
        specificity[class_name] = spec
        print(f"{class_name}: {spec:.4f}")
    
    # ==============================
    # 5️⃣ Model Size
    # ==============================
    temp_path = "temp_model.pth"
    torch.save(model.state_dict(), temp_path)
    model_size = os.path.getsize(temp_path) / (1024 * 1024)
    os.remove(temp_path)
    
    print(f"\nModel Size: {model_size:.2f} MB")
    
    # ==============================
    # 6️⃣ Number of Parameters
    # ==============================
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    
    print(f"Total Parameters: {total_params:,}")
    print(f"Trainable Parameters: {trainable_params:,}")
    
    # ==============================
    # 7️⃣ Inference Time
    # ==============================
    print(f"Inference Time (validation set): {inference_time:.2f} seconds")
    
    print("\n===== Evaluation Complete =====\n")
    
    return {
        "accuracy": accuracy,
        "report": report,
        "confusion_matrix": cm,
        "specificity": specificity,
        "model_size_MB": model_size,
        "total_params": total_params,
        "trainable_params": trainable_params,
        "inference_time": inference_time
    }

### ResNet50

In [ ]:
evaluate_full_model(trained_resnet, dataloaders['val'], class_names, device, model_name="ResNet50")

### YoloV8

In [ ]:
# YoloV8 automatically saves: runs/classify/train/confusion_matrix.png

In [ ]:
model = YOLO("runs/classify/train/weights/best.pt")

metrics = model.val()

print(f"Top-1 Accuracy: {metrics.top1 * 100:.2f}%")
print(f"Top-5 Accuracy: {metrics.top5 * 100:.2f}%")

In [ ]:
test_image_path = 'test.jpg' 

# Run the prediction
results = model(test_image_path)

# Extract and display the results
for result in results:
    top_class_idx = result.probs.top1
    confidence = result.probs.top1conf.item()
    predicted_disease = result.names[top_class_idx]
    
    print(f"Prediction: {predicted_disease}")
    print(f"Confidence: {confidence * 100:.2f}%")

    # Display the image visually with the prediction as the title
    img = Image.open(test_image_path)
    plt.imshow(img)
    plt.title(f"{predicted_disease} ({confidence * 100:.2f}%)")
    plt.axis('off')
    plt.show()

### EfficientNet

In [ ]:
print("Evaluating EfficientNet-B0...")
evaluate_model(trained_efficientnet, dataloaders['val'], class_names, device)